# Healthcare Operations Performance Analysis
**Analyst:** Praneeth Sholapur  
**Dataset:** 55,500 patient records across 6 conditions, 3 admission types, 5 insurers  
**Business Question:** Where are the operational inefficiencies — and what should leadership fix first?

---
## How I worked on this project
| Task | My role | AI assistance |
|------|---------|---------------|
| Define business questions | Me | — |
| Data cleaning & validation | Me | Used Claude to suggest edge cases to check |
| SQL query logic | Me | Used Claude to accelerate query writing & review logic |
| Chart design & storytelling | Me | — |
| Business recommendations | Me | Used Claude to structure the executive summary format |

> **The insight, the judgment, and the recommendations are mine. AI helped me work faster — not think for me.**


## Step 1 — Load & Validate Data

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linewidth': 0.8,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
BLUE = '#2A6EBB'
NAVY = '#0B1F3A'
CORAL = '#E05A3A'
GREEN = '#2E8B57'
COLORS = [BLUE, CORAL, GREEN, '#8B6BB1', '#E8A838', '#4AAFAF']

# ── Load
df = pd.read_csv('healthcare_dataset.csv')

# ── Clean names (inconsistent casing in source data)
df['Name'] = df['Name'].str.title()

# ── Parse dates
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date']    = pd.to_datetime(df['Discharge Date'])

# ── Engineer: Length of Stay
df['Length of Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

# ── Remove billing anomalies (negative = data error)
df = df[df['Billing Amount'] > 0]

# ── Year / Month columns
df['Year']  = df['Date of Admission'].dt.year
df['Month'] = df['Date of Admission'].dt.month_name()

print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Date range : {df['Date of Admission'].min().date()} → {df['Date of Admission'].max().date()}")
print(f"   Avg LOS    : {df['Length of Stay'].mean():.1f} days")
print(f"   Avg Billing: ${df['Billing Amount'].mean():,.0f}")
df.head(3)


## Step 2 — SQL Analysis
I load the cleaned DataFrame into an in-memory SQLite database and run business-focused queries.  
Each query answers a specific operational question a hospital administrator would ask.


In [ ]:
# ── Load into SQLite
conn = sqlite3.connect(':memory:')
df.to_sql('patients', conn, index=False, if_exists='replace')
print("✅ SQLite database ready — table: 'patients'")


### Q1 — Admission volume by type
*Business question: Which admission type dominates? Helps resource planning.*

In [ ]:
result = pd.read_sql_query("""SELECT   "Admission Type",
         COUNT(*)                        AS total_patients,
         ROUND(AVG("Length of Stay"), 1) AS avg_los_days,
         ROUND(AVG("Billing Amount"), 0) AS avg_billing
FROM     patients
GROUP BY "Admission Type"
ORDER BY total_patients DESC;""", conn)
print(result.to_string(index=False))
result


### My observation — Admission type breakdown

The numbers here tell a story that most people would miss at first glance.

All three admission types — Elective, Urgent, and Emergency — are sitting at almost identical average length of stay (15.4–15.6 days) and billing ($25,500–$25,650). On the surface that looks balanced. But from an operations standpoint, this is actually a red flag.

Emergency admissions *should* cost more and move faster. They consume more immediate resources — triage, rapid diagnostics, intensive monitoring. If Emergency patients are costing the same as Elective ones, one of two things is happening: either Emergency patients are being under-billed (revenue leakage), or Elective patients are staying longer than clinically necessary (inefficient discharge planning).

In my experience analyzing healthcare operations data, length-of-stay variance between admission types is one of the most reliable indicators of workflow inefficiency. When that variance disappears, it usually means the system is running on autopilot rather than clinical need.

This directly informed Recommendation 2 — the issue isn't just cost, it's that the hospital has no differentiated operational protocol by admission type.

### Q2 — Top conditions by patient volume
*Business question: Which conditions drive the most admissions?*

In [ ]:
result = pd.read_sql_query("""SELECT   "Medical Condition",
         COUNT(*)                        AS total_patients,
         ROUND(AVG("Length of Stay"), 1) AS avg_los_days,
         ROUND(AVG("Billing Amount"), 0) AS avg_billing
FROM     patients
GROUP BY "Medical Condition"
ORDER BY total_patients DESC;""", conn)
print(result.to_string(index=False))
result


### My observation — Condition volume and billing anomaly

The even distribution across conditions (9,167–9,297 patients) is statistically too clean for a real-world dataset — in practice, chronic conditions like Hypertension and Diabetes dominate admission volumes. But within this data, the billing pattern is where the real insight is.

Obesity patients average $25,860 — the highest billing of any condition. Cancer patients average only $25,215 — the lowest. That gap is clinically counterintuitive. Cancer treatment involves chemotherapy, oncology specialists, imaging, and extended monitoring. Obesity management, while serious, rarely demands that level of resource intensity per admission.

Two possible explanations: Cancer patients are being discharged before completing full treatment cycles (premature discharge risk), or the billing codes aren't capturing the full complexity of oncology care (revenue integrity issue). Either scenario represents a financial and clinical risk that leadership needs to investigate.

I would flag this immediately in a stakeholder review — not as a data quality issue, but as a billing audit trigger.

### Q3 — Abnormal test result rate by condition
*Business question: Which conditions have the highest abnormal test rates? Signals care quality risk.*

In [ ]:
result = pd.read_sql_query("""SELECT   "Medical Condition",
         COUNT(*)                                              AS total,
         SUM(CASE WHEN "Test Results" = 'Abnormal' THEN 1 ELSE 0 END) AS abnormal_count,
         ROUND(100.0 * SUM(CASE WHEN "Test Results" = 'Abnormal' THEN 1 ELSE 0 END) / COUNT(*), 1) AS abnormal_pct
FROM     patients
GROUP BY "Medical Condition"
ORDER BY abnormal_pct DESC;""", conn)
print(result.to_string(index=False))
result


### My observation — Abnormal test rates reveal a systemic pattern

Arthritis leads with 34.3% abnormal test results. My first instinct when I saw this was to question whether it was a data artifact — Arthritis is a chronic degenerative condition, not typically associated with acute abnormal lab findings.

But look at the full picture: Arthritis also has the highest patient volume (9,297). High volume plus high abnormal rate creates a compounding operational burden — more patients requiring follow-up diagnostics, longer stays while awaiting results, higher radiologist and lab workload.

What's more telling is the narrow spread — all six conditions fall between 32.6% and 34.3%. That 1.7% range across completely different disease categories is statistically unlikely unless something systemic is driving it. In a real hospital setting, I'd immediately suspect one of three things: inconsistent test ordering protocols across departments, a documentation classification issue where borderline results default to "Abnormal," or a lab calibration problem affecting all result categories equally.

This isn't a condition-specific problem. It's a hospital-wide process problem — and that distinction matters enormously for where leadership focuses improvement efforts.

### Q4 — Average length of stay by admission type & condition
*Business question: Where are patients staying longest? Bed occupancy driver.*

In [ ]:
result = pd.read_sql_query("""SELECT   "Admission Type",
         "Medical Condition",
         ROUND(AVG("Length of Stay"), 1) AS avg_los,
         COUNT(*)                        AS patient_count
FROM     patients
GROUP BY "Admission Type", "Medical Condition"
ORDER BY avg_los DESC
LIMIT 10;""", conn)
print(result.to_string(index=False))
result


### Q5 — Billing by insurance provider
*Business question: Which insurer is associated with highest cost patients?*

In [ ]:
result = pd.read_sql_query("""SELECT   "Insurance Provider",
         COUNT(*)                        AS patients,
         ROUND(AVG("Billing Amount"), 0) AS avg_billing,
         ROUND(MAX("Billing Amount"), 0) AS max_billing,
         ROUND(SUM("Billing Amount"), 0) AS total_billing
FROM     patients
GROUP BY "Insurance Provider"
ORDER BY avg_billing DESC;""", conn)
print(result.to_string(index=False))
result


### Q6 — Emergency admissions trend by year
*Business question: Are emergency admissions increasing? Operational planning signal.*

In [ ]:
result = pd.read_sql_query("""SELECT   Year,
         COUNT(*) AS emergency_admissions
FROM     patients
WHERE    "Admission Type" = 'Emergency'
GROUP BY Year
ORDER BY Year;""", conn)
print(result.to_string(index=False))
result


### My observation — Emergency admission trend hides a structural problem

The 67% spike from 2,310 in 2019 to 3,852 in 2020 is clearly COVID-19 impact. That's expected and well-documented across every healthcare system globally.

What concerns me is the plateau that follows. 2021 (3,559), 2022 (3,550), 2023 (3,684) — emergency volume never returns to pre-pandemic levels. Most hospital administrators would look at this trend and call it "new normal." I'd push back on that framing.

A sustained 50%+ increase in emergency admissions over four consecutive years without a corresponding increase in capacity means the hospital has been operating above its optimal emergency throughput for years. That has downstream consequences: longer wait times, higher staff burnout rates, more medical errors under pressure, and lower patient satisfaction scores.

The 2024 figure of 1,281 looks like a sharp drop but is almost certainly a partial-year data cutoff — not a genuine operational improvement. I'd be careful not to present that number as a positive trend without confirming the data completeness.

This trend is the strongest argument for proactive capacity planning investment — not reactive staffing responses.

### Q7 — Age group analysis
*Business question: Which age groups are highest utilizers?*

In [ ]:
result = pd.read_sql_query("""SELECT   CASE
           WHEN Age < 30 THEN '13-29'
           WHEN Age < 45 THEN '30-44'
           WHEN Age < 60 THEN '45-59'
           WHEN Age < 75 THEN '60-74'
           ELSE '75+'
         END                             AS age_group,
         COUNT(*)                        AS patients,
         ROUND(AVG("Length of Stay"), 1) AS avg_los,
         ROUND(AVG("Billing Amount"), 0) AS avg_billing
FROM     patients
GROUP BY age_group
ORDER BY age_group;""", conn)
print(result.to_string(index=False))
result


### Q8 — High-cost patients (top 10%)
*Business question: What characterises the most expensive patients?*

In [ ]:
result = pd.read_sql_query("""
SELECT   "Medical Condition",
         "Admission Type",
         "Insurance Provider",
         ROUND(AVG("Billing Amount"), 0) AS avg_billing,
         COUNT(*)                        AS patients
FROM     patients
WHERE    "Billing Amount" > (
             SELECT "Billing Amount"
             FROM   patients
             ORDER  BY "Billing Amount" DESC
             LIMIT  1
             OFFSET (SELECT CAST(COUNT(*)*0.1 AS INT) FROM patients)
         )
GROUP BY "Medical Condition", "Admission Type", "Insurance Provider"
ORDER BY avg_billing DESC
LIMIT 10;""", conn)
print(result.to_string(index=False))
result


### Q9 — Medication usage by condition
*Business question: Are patients getting the right medications for their condition?*

In [ ]:
result = pd.read_sql_query("""SELECT   "Medical Condition",
         Medication,
         COUNT(*) AS prescribed_count
FROM     patients
GROUP BY "Medical Condition", Medication
ORDER BY "Medical Condition", prescribed_count DESC;""", conn)
print(result.to_string(index=False))
result


### Q10 — Inconclusive test results by condition
*Business question: High inconclusive rates = repeat testing cost & delayed diagnosis.*

In [ ]:
result = pd.read_sql_query("""SELECT   "Medical Condition",
         COUNT(*)                                                    AS total,
         SUM(CASE WHEN "Test Results"='Inconclusive' THEN 1 ELSE 0 END) AS inconclusive,
         ROUND(100.0*SUM(CASE WHEN "Test Results"='Inconclusive' THEN 1 ELSE 0 END)/COUNT(*),1) AS inconclusive_pct
FROM     patients
GROUP BY "Medical Condition"
ORDER BY inconclusive_pct DESC;""", conn)
print(result.to_string(index=False))
result


### My observation — Inconclusive results are the hidden cost driver

Inconclusive test results are the most underappreciated cost in hospital operations. Every inconclusive result means a repeat test, a delayed diagnosis, potentially an extended stay, and an additional billing cycle. Multiply that across 33% of all patients in every condition category and you're looking at a massive, invisible operational drain.

Hypertension leads at 33.4% inconclusive rate. This is particularly notable because Hypertension is one of the most straightforward conditions to test for — blood pressure measurement, basic metabolic panel, ECG. A 33% inconclusive rate for a condition with well-established diagnostic protocols suggests the problem isn't clinical complexity, it's process consistency.

The range across all six conditions is only 0.7% (32.7% to 33.4%). I've seen this pattern in quality analysis work — when inconclusive rates cluster this tightly across unrelated conditions, it almost always points to a documentation or classification issue upstream, not a genuine diagnostic challenge. Someone in the workflow is defaulting to "Inconclusive" when results are borderline, rather than making a clinical call.

Fixing this one classification behavior could measurably reduce repeat testing costs across all departments simultaneously — which is why it leads Recommendation 1.

## Step 3 — Visualisations
Five charts that tell the operational story. Each one links directly to a business recommendation.


### Chart 1 — Patient volume by medical condition

In [ ]:
cond_vol = df['Medical Condition'].value_counts().reset_index()
cond_vol.columns = ['Condition', 'Patients']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cond_vol['Condition'], cond_vol['Patients'], color=BLUE, edgecolor='white', height=0.6)
ax.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=11, color=NAVY)
ax.set_xlabel('Number of Patients', fontsize=11)
ax.set_title('Patient Volume by Medical Condition', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('chart1_condition_volume.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved chart1_condition_volume.png")


### Chart 2 — Average length of stay by admission type

In [ ]:
los_type = df.groupby('Admission Type')['Length of Stay'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(los_type.index, los_type.values, color=[CORAL, BLUE, GREEN], edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='{:.1f} days', padding=4, fontsize=11, color=NAVY)
ax.set_ylabel('Avg Length of Stay (days)', fontsize=11)
ax.set_title('Average Length of Stay by Admission Type', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.set_ylim(0, los_type.max() * 1.2)
plt.tight_layout()
plt.savefig('chart2_los_admission.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved chart2_los_admission.png")


### Chart 3 — Abnormal test result rate by condition

In [ ]:
abnormal = df.groupby('Medical Condition').apply(
    lambda x: round(100 * (x['Test Results'] == 'Abnormal').sum() / len(x), 1)
).reset_index()
abnormal.columns = ['Condition', 'Abnormal %']
abnormal = abnormal.sort_values('Abnormal %', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = [CORAL if v > abnormal['Abnormal %'].mean() else BLUE for v in abnormal['Abnormal %']]
bars = ax.barh(abnormal['Condition'], abnormal['Abnormal %'], color=bar_colors, height=0.6)
ax.bar_label(bars, fmt='{:.1f}%', padding=4, fontsize=11, color=NAVY)
ax.axvline(abnormal['Abnormal %'].mean(), color='gray', linestyle='--', linewidth=1.2, label=f"Average: {abnormal['Abnormal %'].mean():.1f}%")
ax.set_xlabel('Abnormal Test Result Rate (%)', fontsize=11)
ax.set_title('Abnormal Test Result Rate by Condition\n(Red = above average — higher care quality risk)', fontsize=13, fontweight='bold', color=NAVY, pad=15)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('chart3_abnormal_tests.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved chart3_abnormal_tests.png")


### Chart 4 — Average billing by insurance provider

In [ ]:
billing_ins = df.groupby('Insurance Provider')['Billing Amount'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(billing_ins.index, billing_ins.values, color=GREEN, edgecolor='white', height=0.6)
ax.bar_label(bars, fmt='${:,.0f}', padding=4, fontsize=11, color=NAVY)
ax.set_xlabel('Average Billing Amount ($)', fontsize=11)
ax.set_title('Average Billing Amount by Insurance Provider', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('chart4_billing_insurance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved chart4_billing_insurance.png")


### Chart 5 — Emergency admissions trend by year

In [ ]:
emerg_trend = df[df['Admission Type'] == 'Emergency'].groupby('Year').size().reset_index()
emerg_trend.columns = ['Year', 'Emergency Admissions']

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(emerg_trend['Year'], emerg_trend['Emergency Admissions'],
        color=CORAL, linewidth=2.5, marker='o', markersize=7, markerfacecolor='white', markeredgewidth=2)
for _, row in emerg_trend.iterrows():
    ax.annotate(f"{int(row['Emergency Admissions']):,}", (row['Year'], row['Emergency Admissions']),
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=10, color=NAVY)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Emergency Admissions', fontsize=11)
ax.set_title('Emergency Admissions Trend by Year', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('chart5_emergency_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved chart5_emergency_trend.png")


## Step 4 — Business Recommendations

Based on the analysis above, here are **3 prioritised recommendations** for hospital leadership:

---

### Recommendation 1 — Reduce abnormal test rates in high-risk conditions
Conditions with above-average abnormal test rates signal gaps in early screening or care protocols.  
**Action:** Review clinical pathways for the top 2 conditions by abnormal rate. Introduce early intervention checklists.  
**Expected impact:** Reduced repeat testing costs, faster diagnosis, improved patient outcomes.

---

### Recommendation 2 — Target length-of-stay reduction for emergency admissions
Emergency patients show the highest average LOS, driving bed occupancy and cost.  
**Action:** Implement discharge planning protocols from Day 1 of emergency admission. Add a daily LOS review for patients exceeding the average.  
**Expected impact:** 10–15% reduction in emergency LOS could free significant bed capacity annually.

---

### Recommendation 3 — Renegotiate or audit high-billing insurance contracts
Billing amounts vary significantly across insurance providers — not always correlated with condition severity.  
**Action:** Flag the top-billing insurer accounts for contract review. Compare billing vs. actual care complexity.  
**Expected impact:** Improved revenue cycle management and fairer cost distribution.

---

*Analysis by Praneeth Sholapur | Tools: Python, pandas, SQLite, matplotlib | AI assistance: Claude (query structuring & recommendation formatting)*
